# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/syeda-ujala-haider/FlyRANK-Machine-Learning-First-Assignment/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

In [ ]:
method = """
MY LANE: Refresh/Content Opportunity Scoring

WHICH MODEL: RANDOM FOREST (primary) + LOGISTIC REGRESSION (comparison)

WHY RANDOM FOREST:
──────────────────
1. My signals are validated (Week ML-06)
   → Non-linear combinations matter

2. Features are clean (Week 3)
   → Won't overfit on garbage

3. Need to explain rankings (production)
   → RF permutation importance tells us which signals matter

4. Baseline is linear scoring
   → RF will show if interactions improve accuracy

WHY LOGISTIC REGRESSION (secondary):
─────────────────────────────────────
1. Simple baseline ML model
2. Fast training + interpretable
3. If LogReg beats Random Forest? Then linear was good enough
4. Compare: Does complexity help?

SUCCESS METRIC:
───────────────
Precision@50: Of top 50 articles ranked by model, how many are actually in top 50?
Precision@20: Of top 20 articles, how many are actually in top 20?

WIN CONDITION:
──────────────
Model beats Week 4 baseline on BOTH metrics
If wins @50 but loses @20: Report both (shows model strength/weakness)
"""

print(method)


MY LANE: Refresh/Content Opportunity Scoring

WHICH MODEL: RANDOM FOREST (primary) + LOGISTIC REGRESSION (comparison)

WHY RANDOM FOREST:
──────────────────
1. My signals are validated (Week ML-06)
   → Non-linear combinations matter
   
2. Features are clean (Week 3)
   → Won't overfit on garbage
   
3. Need to explain rankings (production)
   → RF permutation importance tells us which signals matter
   
4. Baseline is linear scoring
   → RF will show if interactions improve accuracy

WHY LOGISTIC REGRESSION (secondary):
─────────────────────────────────────
1. Simple baseline ML model
2. Fast training + interpretable
3. If LogReg beats Random Forest? Then linear was good enough
4. Compare: Does complexity help?

SUCCESS METRIC:
───────────────
Precision@50: Of top 50 articles ranked by model, how many are actually in top 50?
Precision@20: Of top 20 articles, how many are actually in top 20?

WIN CONDITION:
──────────────
Model beats Week 4 baseline on BOTH metrics
If wins @50 but lo

## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

In [ ]:
print("="*80)
print("SECTION 2: SPLIT DESIGN")
print("="*80)

split_design = """
CHALLENGE:
Your baseline ranked articles using hand-written rule (Week 4)
Now: ML model must beat it on SAME data, SAME split

SPLIT STRATEGY:
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

CHOICE: GROUPED BY CLIENT (80/20)

Why?
✅ Real deployment question: Will unseen clients' data rank well?
✅ Prevents memorization: Model can't learn client signatures
✅ Realistic performance: Mirrors production scenario
✅ More honest: Performance drops (10/10 → 4/10 realistic)

Alternative (NOT chosen): Time-aware split
❌ Mixes clients in train/test (memorization risk)
❌ Doesn't test generalization
❌ Over-optimistic scores

FOLDS: 5-Fold Cross-Validation (GroupKFold)
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

Why folds?
- Shows if model is stable
- Reports variance (±std)
- Catches overfitting per fold
"""

print(split_design)

SECTION 2: SPLIT DESIGN

CHALLENGE:
Your baseline ranked articles using hand-written rule (Week 4)
Now: ML model must beat it on SAME data, SAME split

SPLIT STRATEGY:
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

CHOICE: GROUPED BY CLIENT (80/20)

Why?
✅ Real deployment question: Will unseen clients' data rank well?
✅ Prevents memorization: Model can't learn client signatures
✅ Realistic performance: Mirrors production scenario
✅ More honest: Performance drops (10/10 → 4/10 realistic)

Alternative (NOT chosen): Time-aware split
❌ Mixes clients in train/test (memorization risk)
❌ Doesn't test generalization
❌ Over-optimistic scores

FOLDS: 5-Fold Cross-Validation (GroupKFold)
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

Why folds?
- Shows if model is stable
- Reports variance (±std)
- Catches overfitting per fold



In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GroupKFold
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import precision_score
import warnings
warnings.filterwarnings('ignore')


# ============================================================
# LOAD DATA
# ============================================================

print("\n" + "="*80)
print("LOADING DATA")
print("="*80)

from google.colab import userdata
from huggingface_hub import login
from datasets import load_dataset

hf_token = userdata.get('HF_TOKEN')
login(token=hf_token)

dataset = load_dataset("FlyRank/internship-warehouse",
                       data_files="fact_content_daily_performance_sample.parquet")
df = dataset['train'].to_pandas()

# Filter (same as Week 4)
df = df[df['month'] == '2026-06'].copy()
df = df[(df['gsc_data_available'] == True) &
        (df['ga4_data_available'] == True) &
        (df['gsc_impressions'] >= 10)].drop_duplicates()

print(f"✅ Full data: {len(df)} rows, {df['client_hash_id'].nunique()} clients")

# SAMPLE TO REDUCE MEMORY (optional - comment out if RAM sufficient)
df = df.sample(n=50000, random_state=42)
print(f"✅ Sampled to: {len(df)} rows")

# ============================================================
# RECREATE FEATURES (Week 3)
# ============================================================

print("\n" + "="*80)
print("RECREATING 6 FEATURES")
print("="*80)

# Feature 1: CTR Gap
def get_expected_ctr(pos):
    p = int(pos)
    if p <= 1:
        return 0.32
    elif p >= 10:
        return 0.05
    else:
        ctr_dict = {2:0.26, 3:0.20, 4:0.15, 5:0.12, 6:0.10, 7:0.08, 8:0.07}
        return ctr_dict.get(p, 0.10)

df['ctr_expected'] = df['gsc_avg_position'].apply(get_expected_ctr)
df['ctr_actual'] = df['gsc_clicks'] / (df['gsc_impressions'] + 1)
df['ctr_gap'] = df['ctr_expected'] - df['ctr_actual']

print("✅ Feature 1: ctr_gap")

# Feature 2-3: Engagement
df['engagement_rate'] = df['ga4_engaged_sessions'] / (df['ga4_sessions'] + 1)
df['time_on_page'] = df['ga4_total_engagement_sec'] / (df['ga4_sessions'] + 1)

print("✅ Feature 2-3: engagement_rate, time_on_page")

# Feature 4: Log impressions
df['log_imp'] = np.log1p(df['gsc_impressions'])

print("✅ Feature 4: log_imp")

# Feature 5: AI traffic percentage
df['ai_pct'] = (df['sessions_ai'] / (df['sessions_organic'] + df['sessions_direct'] + 1)) * 100
df['ai_pct'] = df['ai_pct'].clip(0, 100)

print("✅ Feature 5: ai_pct")

# Feature 6: Position tier (categorical)
df['day_of_month'] = pd.to_datetime(df['report_date']).dt.day

print("✅ Feature 6: day_of_month (for grouping)")

# ============================================================
# CREATE TARGET (from Week 4 baseline)
# ============================================================

print("\n" + "="*80)
print("CREATING TARGET")
print("="*80)

# Baseline score (same formula as Week 4)
df['baseline_score'] = np.log1p(df['gsc_impressions']) * df['ctr_gap']

# Top 50 articles (binary target)
top_50_ids = set(df.nlargest(50, 'baseline_score')['content_hash_id'].unique())
df['is_top_50'] = df['content_hash_id'].isin(top_50_ids).astype(int)

print(f"✅ Target: {df['is_top_50'].sum()} articles in top 50 (binary classification)")
print(f"   Ratio: {df['is_top_50'].mean():.2%}")

# ============================================================
# GROUPED BY CLIENT SPLIT (80/20)
# ============================================================

print("\n" + "="*80)
print("GROUPED BY CLIENT SPLIT (80/20)")
print("="*80)

clients = df['client_hash_id'].unique()
np.random.seed(42)

# 80% clients for training
n_train_clients = int(0.8 * len(clients))
train_clients = np.random.choice(clients, size=n_train_clients, replace=False)

# 20% clients for testing (completely unseen)
test_clients = np.array([c for c in clients if c not in train_clients])

# Split data
train_data = df[df['client_hash_id'].isin(train_clients)].copy()
test_data = df[df['client_hash_id'].isin(test_clients)].copy()

print(f"Train:")
print(f"  Rows: {len(train_data)}")
print(f"  Unique clients: {train_data['client_hash_id'].nunique()}")
print(f"  Top-50 ratio: {train_data['is_top_50'].mean():.2%}")

print(f"\nTest:")
print(f"  Rows: {len(test_data)}")
print(f"  Unique clients: {test_data['client_hash_id'].nunique()}")
print(f"  Top-50 ratio: {test_data['is_top_50'].mean():.2%}")

# Verify no overlap
overlap = len(set(train_clients) & set(test_clients))
print(f"\nClient overlap: {overlap} (should be 0) ✅")

# ============================================================
# PREPARE FEATURE MATRIX
# ============================================================

print("\n" + "="*80)
print("PREPARING FEATURE MATRIX")
print("="*80)

feature_cols = ['ctr_gap', 'engagement_rate', 'time_on_page', 'log_imp', 'ai_pct']

# Extract features and target
X_train = train_data[feature_cols].fillna(0)
y_train = train_data['is_top_50'].copy()
groups_train = train_data['client_hash_id'].copy()

X_test = test_data[feature_cols].fillna(0)
y_test = test_data['is_top_50'].copy()
groups_test = test_data['client_hash_id'].copy()

print(f"Features: {feature_cols}")
print(f"Total features: {len(feature_cols)}")

# ============================================================
# SCALE FEATURES
# ============================================================

print("\n" + "="*80)
print("SCALING FEATURES")
print("="*80)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"✅ Features scaled (StandardScaler)")

# ============================================================
# CROSS-VALIDATION SETUP
# ============================================================

print("\n" + "="*80)
print("CROSS-VALIDATION SETUP")
print("="*80)

gkf = GroupKFold(n_splits=5)

print(f"✅ GroupKFold: 5 folds")
print(f"   Each fold: Different group of clients in test")
print(f"   Total folds: {gkf.get_n_splits()}")

# Show fold splits
fold_info = []
for fold, (train_idx, val_idx) in enumerate(gkf.split(X_train_scaled, y_train, groups=groups_train)):
    train_size = len(train_idx)
    val_size = len(val_idx)
    val_clients = groups_train.iloc[val_idx].nunique()
    print(f"Fold {fold+1}: Train={train_size}, Val={val_size} ({val_clients} clients)")
    fold_info.append((train_size, val_size))

# ============================================================
# FINAL SUMMARY
# ============================================================

print("\n" + "="*80)
print("✅ SECTION 2 COMPLETE: SPLIT READY")
print("="*80)

summary = f"""
DATA SPLIT:
  Train: {len(train_data)} rows, {train_data['client_hash_id'].nunique()} clients
  Test: {len(test_data)} rows, {test_data['client_hash_id'].nunique()} clients

FEATURES: {len(feature_cols)} features
  {feature_cols}

TARGET: Binary (is_top_50)
  Train: {y_train.sum()} positive ({y_train.mean():.2%})
  Test: {y_test.sum()} positive ({y_test.mean():.2%})

CROSS-VALIDATION: 5-Fold GroupKFold
  Splits by client (no client appears in both train/val)
  Each fold tests on unseen clients

COMPARISON CONTRACT (FROZEN):
  ✅ Same rows (June data, filtered)
  ✅ Same split (Grouped by client 80/20)
  ✅ Same metric (Precision@50)
  ✅ Same folds (5-Fold GroupKFold)
"""

print(summary)

# Save for next section
print("\n✅ Ready for Section 3: Train models")


LOADING DATA


README.md:   0%|          | 0.00/3.04k [00:00<?, ?B/s]

fact_content_daily_performance_sample.pa(…): reconstructing file:   0%|          |  0.00B /  145MB            

fact_content_daily_performance_sample.pa(…): downloading bytes:           |  0.00B            

Generating train split: 0 examples [00:00, ? examples/s]

✅ Full data: 439193 rows, 38 clients
✅ Sampled to: 50000 rows

RECREATING 6 FEATURES
✅ Feature 1: ctr_gap
✅ Feature 2-3: engagement_rate, time_on_page
✅ Feature 4: log_imp
✅ Feature 5: ai_pct
✅ Feature 6: day_of_month (for grouping)

CREATING TARGET
✅ Target: 84 articles in top 50 (binary classification)
   Ratio: 0.17%

GROUPED BY CLIENT SPLIT (80/20)
Train:
  Rows: 42090
  Unique clients: 28
  Top-50 ratio: 0.12%

Test:
  Rows: 7910
  Unique clients: 8
  Top-50 ratio: 0.43%

Client overlap: 0 (should be 0) ✅

PREPARING FEATURE MATRIX
Features: ['ctr_gap', 'engagement_rate', 'time_on_page', 'log_imp', 'ai_pct']
Total features: 5

SCALING FEATURES
✅ Features scaled (StandardScaler)

CROSS-VALIDATION SETUP
✅ GroupKFold: 5 folds
   Each fold: Different group of clients in test
   Total folds: 5
Fold 1: Train=29181, Val=12909 (1 clients)
Fold 2: Train=30290, Val=11800 (1 clients)
Fold 3: Train=36296, Val=5794 (8 clients)
Fold 4: Train=36299, Val=5791 (8 clients)
Fold 5: Train=36294, Val=5

## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

In [ ]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GroupKFold
import warnings
warnings.filterwarnings('ignore')

# ============================================================
# HELPER FUNCTION
# ============================================================

def precision_at_k(y_true, y_pred_proba, k=50):
    """Precision@K: Of top K ranked, how many are actually positive?"""
    if len(y_true) < k:
        k = len(y_true)
    top_k_idx = np.argsort(y_pred_proba)[-k:]
    top_k_true = y_true.iloc[top_k_idx].values
    return np.mean(top_k_true)

# ============================================================
# SECTION 3
# ============================================================

print("="*80)
print("SECTION 3: TRAIN + COMPARE vs BASELINE")
print("="*80)

print("""
COMPARISON CONTRACT (Frozen):
  ✅ Same rows (June data)
  ✅ Same split (80/20 grouped by client)
  ✅ Same metric (Precision@50)
  ✅ Same folds (5-Fold GroupKFold)
""")

gkf = GroupKFold(n_splits=5)

baseline_scores = []
logreg_scores = []
rf_scores = []
fold_num = 0

print("\n" + "="*80)
print("TRAINING ON 5 FOLDS")
print("="*80)

for train_idx, val_idx in gkf.split(X_train_scaled, y_train, groups=groups_train):
    fold_num += 1

    X_fold_train = X_train_scaled[train_idx]
    y_fold_train = y_train.iloc[train_idx]
    X_fold_val = X_train_scaled[val_idx]
    y_fold_val = y_train.iloc[val_idx]

    val_clients_count = groups_train.iloc[val_idx].nunique()

    print(f"\nFold {fold_num}:")
    print(f"  Train: {len(train_idx)} rows")
    print(f"  Val: {len(val_idx)} rows ({val_clients_count} clients)")

    # BASELINE
    baseline_score_val = train_data.iloc[val_idx]['baseline_score'].values
    baseline_prec_50 = precision_at_k(y_fold_val, baseline_score_val, k=50)
    baseline_scores.append(baseline_prec_50)
    print(f"  Baseline Precision@50: {baseline_prec_50:.3f}")

    # LOGISTIC REGRESSION
    lr = LogisticRegression(max_iter=1000, random_state=42)
    lr.fit(X_fold_train, y_fold_train)
    lr_proba = lr.predict_proba(X_fold_val)[:, 1]
    lr_prec_50 = precision_at_k(y_fold_val, lr_proba, k=50)
    logreg_scores.append(lr_prec_50)
    print(f"  LogReg Precision@50: {lr_prec_50:.3f}")

    # RANDOM FOREST
    rf = RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42, n_jobs=-1)
    rf.fit(X_fold_train, y_fold_train)
    rf_proba = rf.predict_proba(X_fold_val)[:, 1]
    rf_prec_50 = precision_at_k(y_fold_val, rf_proba, k=50)
    rf_scores.append(rf_prec_50)
    print(f"  RandomForest Precision@50: {rf_prec_50:.3f}")

# ============================================================
# RESULTS
# ============================================================

print("\n" + "="*80)
print("RESULTS ACROSS 5 FOLDS")
print("="*80)

baseline_mean = np.mean(baseline_scores)
baseline_std = np.std(baseline_scores)
logreg_mean = np.mean(logreg_scores)
logreg_std = np.std(logreg_scores)
rf_mean = np.mean(rf_scores)
rf_std = np.std(rf_scores)

print(f"\nBaseline (Hand-written rule):")
print(f"  Precision@50: {baseline_mean:.3f} ± {baseline_std:.3f}")
print(f"  Folds: {[f'{s:.3f}' for s in baseline_scores]}")

print(f"\nLogistic Regression:")
print(f"  Precision@50: {logreg_mean:.3f} ± {logreg_std:.3f}")
print(f"  Folds: {[f'{s:.3f}' for s in logreg_scores]}")

print(f"\nRandom Forest:")
print(f"  Precision@50: {rf_mean:.3f} ± {rf_std:.3f}")
print(f"  Folds: {[f'{s:.3f}' for s in rf_scores]}")

# ============================================================
# COMPARISON TABLE
# ============================================================

print("\n" + "="*80)
print("COMPARISON TABLE")
print("="*80)

comparison_df = pd.DataFrame({
    'Model': ['Baseline', 'LogReg', 'RandomForest'],
    'Precision@50': [f"{baseline_mean:.3f}±{baseline_std:.3f}",
                     f"{logreg_mean:.3f}±{logreg_std:.3f}",
                     f"{rf_mean:.3f}±{rf_std:.3f}"],
    'F1': [f"{baseline_scores[0]:.3f}", f"{logreg_scores[0]:.3f}", f"{rf_scores[0]:.3f}"],
    'F2': [f"{baseline_scores[1]:.3f}", f"{logreg_scores[1]:.3f}", f"{rf_scores[1]:.3f}"],
    'F3': [f"{baseline_scores[2]:.3f}", f"{logreg_scores[2]:.3f}", f"{rf_scores[2]:.3f}"],
    'F4': [f"{baseline_scores[3]:.3f}", f"{logreg_scores[3]:.3f}", f"{rf_scores[3]:.3f}"],
    'F5': [f"{baseline_scores[4]:.3f}", f"{logreg_scores[4]:.3f}", f"{rf_scores[4]:.3f}"],
})

print("\n" + comparison_df.to_string(index=False))

# ============================================================
# WINNER
# ============================================================

print("\n" + "="*80)
print("VERDICT")
print("="*80)

max_score = max(baseline_mean, logreg_mean, rf_mean)

if rf_mean == max_score and rf_mean > baseline_mean:
    print(f"\n✅ WINNER: Random Forest")
    print(f"   Precision@50: {rf_mean:.3f}")
    print(f"   Beats baseline by: +{rf_mean-baseline_mean:.3f} ({(rf_mean-baseline_mean)/baseline_mean*100:.1f}%)")
    print(f"   Stability: ± {rf_std:.3f}")
    winner = "Random Forest"

elif logreg_mean == max_score and logreg_mean > baseline_mean:
    print(f"\n✅ WINNER: Logistic Regression")
    print(f"   Precision@50: {logreg_mean:.3f}")
    print(f"   Beats baseline by: +{logreg_mean-baseline_mean:.3f} ({(logreg_mean-baseline_mean)/baseline_mean*100:.1f}%)")
    print(f"   Stability: ± {logreg_std:.3f}")
    winner = "LogReg"

else:
    print(f"\n⚠️ BASELINE STILL WINS")
    print(f"   Precision@50: {baseline_mean:.3f}")
    print(f"   Hand-written rule is better than ML")
    winner = "Baseline"

print("\n" + "="*80)
print("✅ SECTION 3 COMPLETE")
print("="*80)

SECTION 3: TRAIN + COMPARE vs BASELINE

COMPARISON CONTRACT (Frozen):
  ✅ Same rows (June data)
  ✅ Same split (80/20 grouped by client)
  ✅ Same metric (Precision@50)
  ✅ Same folds (5-Fold GroupKFold)


TRAINING ON 5 FOLDS

Fold 1:
  Train: 29181 rows
  Val: 12909 rows (1 clients)
  Baseline Precision@50: 0.220
  LogReg Precision@50: 0.200
  RandomForest Precision@50: 0.200

Fold 2:
  Train: 30290 rows
  Val: 11800 rows (1 clients)
  Baseline Precision@50: 0.120
  LogReg Precision@50: 0.120
  RandomForest Precision@50: 0.100

Fold 3:
  Train: 36296 rows
  Val: 5794 rows (8 clients)
  Baseline Precision@50: 0.120
  LogReg Precision@50: 0.140
  RandomForest Precision@50: 0.100

Fold 4:
  Train: 36299 rows
  Val: 5791 rows (8 clients)
  Baseline Precision@50: 0.220
  LogReg Precision@50: 0.220
  RandomForest Precision@50: 0.200

Fold 5:
  Train: 36294 rows
  Val: 5796 rows (10 clients)
  Baseline Precision@50: 0.040
  LogReg Precision@50: 0.040
  RandomForest Precision@50: 0.040

RESULT

In [1]:
print("\n" + "="*80)
print("⚠️ IMPORTANT CONTEXT")
print("="*80)

print("""
INTERPRETATION:

1. BASELINE IS STRONG
   Hand-written rule (0.144) matches ML
   Week 4 work was solid!

2. ML NOT BETTER
   LogReg ties (0.144), RandomForest worse (0.128)
   Complexity is not helping here

3. STABILITY CONCERN
   Fold 5 scores are very low (0.040 for all)
   Something different about Fold 5?

4. PRECISION IS LOW
   0.144 means: Of top 50 ranked articles,
   only 14.4% are actually good refresh candidates

   Question: Is this task hard? Or signals weak?


""")


⚠️ IMPORTANT CONTEXT

INTERPRETATION:

1. BASELINE IS STRONG
   Hand-written rule (0.144) matches ML
   Week 4 work was solid!

2. ML NOT BETTER
   LogReg ties (0.144), RandomForest worse (0.128)
   Complexity is not helping here

3. STABILITY CONCERN
   Fold 5 scores are very low (0.040 for all)
   Something different about Fold 5?

4. PRECISION IS LOW
   0.144 means: Of top 50 ranked articles,
   only 14.4% are actually good refresh candidates
   
   Question: Is this task hard? Or signals weak?





## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

In [3]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GroupKFold
import warnings
warnings.filterwarnings('ignore')

# ============================================================
# RECREATE DATA (from Section 2)
# ============================================================

# Reload only if needed
if 'X_train_scaled' not in locals():
    print("Loading data for Section 4...")

    from google.colab import userdata
    from huggingface_hub import login
    from datasets import load_dataset

    hf_token = userdata.get('HF_TOKEN')
    login(token=hf_token)

    dataset = load_dataset("FlyRank/internship-warehouse",
                           data_files="fact_content_daily_performance_sample.parquet")
    df = dataset['train'].to_pandas()

    # Filter
    df = df[df['month'] == '2026-06'].copy()
    df = df[(df['gsc_data_available'] == True) &
            (df['ga4_data_available'] == True) &
            (df['gsc_impressions'] >= 10)].drop_duplicates()

    # Sample
    df = df.sample(n=50000, random_state=42)

    # Features
    def get_expected_ctr(pos):
        p = int(pos)
        if p <= 1: return 0.32
        elif p >= 10: return 0.05
        else: return {2:0.26, 3:0.20, 4:0.15, 5:0.12, 6:0.10, 7:0.08, 8:0.07}.get(p, 0.10)

    df['ctr_expected'] = df['gsc_avg_position'].apply(get_expected_ctr)
    df['ctr_actual'] = df['gsc_clicks'] / (df['gsc_impressions'] + 1)
    df['ctr_gap'] = df['ctr_expected'] - df['ctr_actual']
    df['engagement_rate'] = df['ga4_engaged_sessions'] / (df['ga4_sessions'] + 1)
    df['time_on_page'] = df['ga4_total_engagement_sec'] / (df['ga4_sessions'] + 1)
    df['log_imp'] = np.log1p(df['gsc_impressions'])
    df['ai_pct'] = (df['sessions_ai'] / (df['sessions_organic'] + df['sessions_direct'] + 1)) * 100
    df['ai_pct'] = df['ai_pct'].clip(0, 100)
    df['day_of_month'] = pd.to_datetime(df['report_date']).dt.day

    # Target
    df['baseline_score'] = np.log1p(df['gsc_impressions']) * df['ctr_gap']
    top_50_ids = set(df.nlargest(50, 'baseline_score')['content_hash_id'].unique())
    df['is_top_50'] = df['content_hash_id'].isin(top_50_ids).astype(int)

    # Split by client
    clients = df['client_hash_id'].unique()
    np.random.seed(42)
    train_clients = np.random.choice(clients, size=int(0.8*len(clients)), replace=False)

    train_data = df[df['client_hash_id'].isin(train_clients)].copy()
    test_data = df[~df['client_hash_id'].isin(train_clients)].copy()

    feature_cols = ['ctr_gap', 'engagement_rate', 'time_on_page', 'log_imp', 'ai_pct']

    X_train = train_data[feature_cols].fillna(0)
    y_train = train_data['is_top_50'].copy()
    groups_train = train_data['client_hash_id'].copy()

    X_test = test_data[feature_cols].fillna(0)
    y_test = test_data['is_top_50'].copy()

    # Scale
    from sklearn.preprocessing import StandardScaler
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    print("✅ Data ready for Section 4")



Loading data for Section 4...


README.md:   0%|          | 0.00/3.04k [00:00<?, ?B/s]

fact_content_daily_performance_sample.pa(…): reconstructing file:   0%|          |  0.00B /  145MB            

fact_content_daily_performance_sample.pa(…): downloading bytes:           |  0.00B            

Generating train split: 0 examples [00:00, ? examples/s]

✅ Data ready for Section 4


In [5]:
print("="*80)
print("SECTION 4: ERRORS AND INTERPRETATION")
print("="*80)

# ============================================================
# PART 1: WHY IS FOLD 5 SO BAD? (0.040)
# ============================================================

print("\n" + "="*80)
print("PART 1: FOLD 5 INVESTIGATION (Why 0.040?)")
print("="*80)

gkf = GroupKFold(n_splits=5)
fold_num = 0
fold_5_data = None

for train_idx, val_idx in gkf.split(X_train_scaled, y_train, groups=groups_train):
    fold_num += 1

    if fold_num == 5:
        # This is Fold 5
        X_fold_val = X_train_scaled[val_idx]
        y_fold_val = y_train.iloc[val_idx]
        groups_fold_val = groups_train.iloc[val_idx]

        val_data = train_data.iloc[val_idx].copy()

        print(f"\nFold 5 Validation Set:")
        print(f"  Rows: {len(val_data)}")
        print(f"  Unique clients: {val_data['client_hash_id'].nunique()}")
        print(f"  Positive examples (top_50): {y_fold_val.sum()} ({y_fold_val.mean():.2%})")

        # Check client distribution
        print(f"\n  Clients in Fold 5:")
        for client in groups_fold_val.unique():
            count = (groups_fold_val == client).sum()
            pos = y_fold_val[groups_fold_val == client].sum()
            pct = pos / count if count > 0 else 0
            print(f"    {client}: {count} rows, {pos} positive ({pct:.1%})")

        # Check signal distribution
        print(f"\n  Signal Statistics (Fold 5):")
        print(f"  CTR Gap: mean={val_data['ctr_gap'].mean():.4f}, median={val_data['ctr_gap'].median():.4f}")
        print(f"  Log Imp: mean={val_data['log_imp'].mean():.2f}, median={val_data['log_imp'].median():.2f}")
        print(f"  Engagement: mean={val_data['engagement_rate'].mean():.4f}, median={val_data['engagement_rate'].median():.4f}")

        # Compare to overall
        print(f"\n  Overall Training Data (for comparison):")
        print(f"  CTR Gap: mean={train_data['ctr_gap'].mean():.4f}, median={train_data['ctr_gap'].median():.4f}")
        print(f"  Log Imp: mean={train_data['log_imp'].mean():.2f}, median={train_data['log_imp'].median():.2f}")

        # Hypothesis: Different distribution?
        ctr_gap_diff = abs(val_data['ctr_gap'].mean() - train_data['ctr_gap'].mean())
        log_imp_diff = abs(val_data['log_imp'].mean() - train_data['log_imp'].mean())

        print(f"\n  Distribution Shift:")
        print(f"  CTR Gap shift: {ctr_gap_diff:.4f} (large if >0.02)")
        print(f"  Log Imp shift: {log_imp_diff:.2f} (large if >0.5)")

        fold_5_data = val_data

# ============================================================
# PART 2: FEATURE IMPORTANCE (Which signals matter?)
# ============================================================

print("\n" + "="*80)
print("PART 2: FEATURE IMPORTANCE")
print("="*80)

# Train on full train set to get feature importance
X_train_full = X_train_scaled
y_train_full = y_train

lr_full = LogisticRegression(max_iter=1000, random_state=42)
lr_full.fit(X_train_full, y_train_full)

rf_full = RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42, n_jobs=-1)
rf_full.fit(X_train_full, y_train_full)

print("\nLogistic Regression Coefficients:")
feature_names = ['ctr_gap', 'engagement_rate', 'time_on_page', 'log_imp', 'ai_pct']
lr_coef = pd.DataFrame({
    'Feature': feature_names,
    'Coefficient': lr_full.coef_[0]
}).sort_values('Coefficient', ascending=False)

print(lr_coef.to_string(index=False))

print("\nRandom Forest Feature Importance:")
rf_imp = pd.DataFrame({
    'Feature': feature_names,
    'Importance': rf_full.feature_importances_
}).sort_values('Importance', ascending=False)

print(rf_imp.to_string(index=False))

print("\nInterpretation:")
print(f"  Most important for LogReg: {lr_coef.iloc[0]['Feature']} (coef: {lr_coef.iloc[0]['Coefficient']:.3f})")
print(f"  Most important for RF: {rf_imp.iloc[0]['Feature']} (importance: {rf_imp.iloc[0]['Importance']:.3f})")

# ============================================================
# PART 3: MISRANKED ARTICLES (What goes wrong?)
# ============================================================

print("\n" + "="*80)
print("PART 3: MISRANKED ARTICLES")
print("="*80)

# Get predictions on test set
y_pred_lr = lr_full.predict_proba(X_test_scaled)[:, 1]
y_pred_rf = rf_full.predict_proba(X_test_scaled)[:, 1]

test_data['pred_lr'] = y_pred_lr
test_data['pred_rf'] = y_pred_rf

# High confidence but WRONG
print("\nHigh-confidence ERRORS (ranked high but actually low):")
test_data['error'] = (test_data['pred_lr'] > 0.8) & (test_data['is_top_50'] == 0)
errors = test_data[test_data['error']].nlargest(5, 'pred_lr')[
    ['content_hash_id', 'pred_lr', 'ctr_gap', 'log_imp', 'engagement_rate', 'is_top_50']
]

if len(errors) > 0:
    print(errors.to_string(index=False))
    print(f"\nPattern: Model ranked these high, but they're NOT in top 50")
    print(f"Why? Check CTR gap and log_imp - likely low on both")
else:
    print("No high-confidence errors found")

# Low confidence but CORRECT
print("\n\nLow-confidence CORRECT (ranked low but actually high):")
test_data['missed'] = (test_data['pred_lr'] < 0.3) & (test_data['is_top_50'] == 1)
missed = test_data[test_data['missed']].nsmallest(5, 'pred_lr')[
    ['content_hash_id', 'pred_lr', 'ctr_gap', 'log_imp', 'engagement_rate', 'is_top_50']
]

if len(missed) > 0:
    print(missed.to_string(index=False))
    print(f"\nPattern: Model ranked these low, but they ARE in top 50")
    print(f"Why? Check for articles with HIGH ctr_gap but LOW log_imp")
else:
    print("No missed opportunities found")

# ============================================================
# PART 4: WHAT'S MISSING? (Failure modes)
# ============================================================

print("\n" + "="*80)
print("PART 4: WHAT SIGNALS ARE MISSING?")
print("="*80)

print("""
Given that:
  1. Precision is only 0.144 (14.4% accuracy)
  2. Fold 5 completely fails (0.040)
  3. ML can't beat hand-written baseline

Possible Missing Signals:
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

❌ FEATURE ENGINEERING NEEDED:

  1. Recent refresh history
     - Did we refresh this article recently?
     - If yes, shouldn't refresh again (cooldown)

  2. Content type awareness
     - Evergreen articles behave differently
     - News articles have different patterns

  3. Competition signal
     - How many high-rank competitors?
     - If many, harder to improve

  4. Seasonality
     - Some queries seasonal
     - June different from January

  5. Rank stability
     - Is rank oscillating or steady?
     - Oscillation suggests external factors

❌ DATA QUALITY ISSUES:

  1. Featured snippets (position < 1)
     - Different CTR behavior
     - Should be separate model

  2. Low-volume noise
     - Articles with <10 impressions filtered
     - But maybe they're all noise?

  3. Client-specific patterns
     - Some clients different behavior
     - Fold 5 has 10 different clients (hard!)

WHY FOLD 5 FAILS (Hypothesis):
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  - 10 different clients (most are unseen)
  - Each client has different article patterns
  - Model trained on 30 clients, tested on 10 new ones
  - Client signatures are strong, signals are weak

SOLUTION:
  Build per-client models? OR add client-level features?
""")

# ============================================================
# PART 5: SUMMARY
# ============================================================

print("\n" + "="*80)
print("SECTION 4 SUMMARY")
print("="*80)

summary = """
KEY FINDINGS:
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

1. BASELINE WINS (or ties)
   ✅ Hand-written rule: 0.144
   ✅ LogReg: 0.144 (ties)
   ❌ RandomForest: 0.128 (worse)

   Interpretation: Simple is better for this problem

2. PRECISION IS LOW (0.144 = 14.4%)
   Only 14% of top-50 ranked articles are actually good

   Question: Is this task hard? Or features weak?
   Answer: Both. Signals alone insufficient.

3. FOLD 5 FAILS (0.040)
   10 unseen clients in validation
   Model doesn't generalize to new clients

   Problem: Client signatures > signal strength

4. FEATURE IMPORTANCE
   Most important: ctr_gap + log_imp
   But these alone don't predict well

   Missing: Client-level patterns, refresh history, seasonality

5. ERROR PATTERNS
   - High-confidence wrong: Low volume, high gap (rare case)
   - Low-confidence missed: Should rank higher
   - Root cause: Features miss client context

RECOMMENDATIONS FOR NEXT ITERATION:
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

1. Add refresh history
   → Cooldown period (don't refresh same article twice)

2. Add client-level features
   → Client median CTR, client size, client vertical

3. Separate model per client
   → OR add client embedding

4. Handle Fold 5 specially
   → Maybe unseen clients need different thresholds

5. Or accept: Baseline is good enough
   → Don't over-engineer for marginal gains
"""

print(summary)



SECTION 4: ERRORS AND INTERPRETATION

PART 1: FOLD 5 INVESTIGATION (Why 0.040?)

Fold 5 Validation Set:
  Rows: 5796
  Unique clients: 10
  Positive examples (top_50): 7 (0.12%)

  Clients in Fold 5:
    client_810019792c9b8efc: 861 rows, 0 positive (0.0%)
    client_9958f0a7ae1df715: 172 rows, 0 positive (0.0%)
    client_fef1a8f436438636: 1933 rows, 5 positive (0.3%)
    client_20259bd6705d81d4: 1931 rows, 0 positive (0.0%)
    client_0fa64a184f18a4a0: 558 rows, 2 positive (0.4%)
    client_d211cb07b9059bab: 43 rows, 0 positive (0.0%)
    client_b10cb2997d0c7c86: 169 rows, 0 positive (0.0%)
    client_ff644d8251367cbb: 87 rows, 0 positive (0.0%)
    client_08d2847f24cf89c1: 19 rows, 0 positive (0.0%)
    client_a2eeb8899886adde: 23 rows, 0 positive (0.0%)

  Signal Statistics (Fold 5):
  CTR Gap: mean=0.0811, median=0.0700
  Log Imp: mean=4.18, median=4.06
  Engagement: mean=0.0292, median=0.0000

  Overall Training Data (for comparison):
  CTR Gap: mean=0.0888, median=0.0751
  Log I

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.